# Evaluate M³ fast against M³ base

Compares M³ fast, the previous M³ run, and ReAct. M³ base and ReAct are Llama3 70B (`default-text-large`) and Qwen 3.6 35B (`Qwen_Qwen3.6-35B-A3B-FP8`). M³ fast also includes Qwen 3.5 122B (`m3_fast_all_Qwen3.5-122B-A10B-NVFP4`) and Mistral Devstral 2 24B (`m3_fast_all_Devstral-Small-2-24B-Instruct-2512`). The samples are all 75 PANGAEA datasets and all 100 iNaturalist species.

Spatial and temporal scoring is the same as `07-TvdP-evaluate-technical-autoddg.ipynb`. PANGAEA true-positive accuracy is the mean number of exact matches (`numpy.isclose` for coordinates, same calendar day for dates). Precision and recall treat a filled field as a positive. iNaturalist accuracy is an exact match on each bound and timestamp; with 100 species that count is the percent. A missing file is reported and does not count as a match. On PANGAEA, an unvisited dataset is left out of precision, recall, and the match count.

The first code cell prints technical scores only. The next cell scores PANGAEA descriptions. M³ fast descriptions are judged and checkpointed. M³ base and ReAct descriptions are the scores already stored in `df_scores_pangaea_descriptive_with_baselines.csv`. The last cell prints technical and descriptive scores together. iNaturalist files for these runs contain spatial and temporal coverage only, so descriptive scores are PANGAEA scores.


In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path('/home/com3dian/Github/process_data_metadata_agent')
M3_ROOT = Path('/home/com3dian/Github/metadata_agent/outputs')
INAT_DATA = REPO / 'outputs' / 'inaturalist' / 'inaturalist_100_species_1000_obs'
PANGAEA_ANNOT = REPO / 'outputs' / 'pangaea' / 'sample_20260629_163144_filtered_top5_per_topic.csv'
PANGAEA_EXTENT_CACHE = REPO / 'outputs' / 'pangaea' / 'sample_20260629_163144_extents.json'
DESCRIPTIVE_SCORES = REPO / 'outputs' / 'pangaea' / 'df_scores_pangaea_descriptive_with_baselines.csv'
OUT = REPO / 'outputs' / 'm3_fast_vs_base'
OUT.mkdir(parents=True, exist_ok=True)

# Previous M³ descriptive scores were stored under the Llama3 70B column.
# default-text-large is that model.
M3_BASE_FAITH_COLUMN = 'faith_llama3_70B'
M3_BASE_SIM_COLUMN = 'sim_llama3_70B'

LLAMA = 'Llama3 70B'
QWEN = 'Qwen 3.6 35B'
QWEN35 = 'Qwen 3.5 122B'
DEVSTRAL = 'Mistral Devstral 2 24B'
METHODS = [
    {
        'method': 'M³ base',
        'llm': LLAMA,
        'pangaea': M3_ROOT / 'pangaea_inat_default-text-large',
        'inaturalist': M3_ROOT / 'inat_default-text-large',
    },
    {
        'method': 'M³ base',
        'llm': QWEN,
        'pangaea': M3_ROOT / 'pangaea_inat_Qwen_Qwen3.6-35B-A3B-FP8',
        'inaturalist': M3_ROOT / 'inat_Qwen_Qwen3.6-35B-A3B-FP8',
    },
    {
        'method': 'M³ fast',
        'llm': LLAMA,
        'pangaea': M3_ROOT / 'm3_fast_all_default-text-large_recovered' / 'm3' / 'pangaea',
        'inaturalist': M3_ROOT / 'm3_fast_all_default-text-large_recovered' / 'm3' / 'inaturalist',
    },
    {
        'method': 'M³ fast',
        'llm': QWEN,
        'pangaea': M3_ROOT / 'm3_fast_all_Qwen3.6-35B-A3B-FP8' / 'm3' / 'pangaea',
        'inaturalist': M3_ROOT / 'm3_fast_all_Qwen3.6-35B-A3B-FP8' / 'm3' / 'inaturalist',
    },
    {
        'method': 'M³ fast',
        'llm': QWEN35,
        'pangaea': M3_ROOT / 'm3_fast_all_Qwen3.5-122B-A10B-NVFP4' / 'm3' / 'pangaea',
        'inaturalist': M3_ROOT / 'm3_fast_all_Qwen3.5-122B-A10B-NVFP4' / 'm3' / 'inaturalist',
    },
    {
        'method': 'M³ fast',
        'llm': DEVSTRAL,
        'pangaea': M3_ROOT / 'm3_fast_all_Devstral-Small-2-24B-Instruct-2512' / 'm3' / 'pangaea',
        'inaturalist': M3_ROOT / 'm3_fast_all_Devstral-Small-2-24B-Instruct-2512' / 'm3' / 'inaturalist',
    },
    {
        'method': 'ReAct',
        'llm': LLAMA,
        'pangaea': M3_ROOT / 'react_pangaea_default-text-large',
        'inaturalist': M3_ROOT / 'react_inaturalist_default-text-large',
    },
    {
        'method': 'ReAct',
        'llm': QWEN,
        'pangaea': M3_ROOT / 'react_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
        'inaturalist': M3_ROOT / 'react_inaturalist_Qwen_Qwen3.6-35B-A3B-FP8',
    },
]


def fix_dt(dt_str):
    """Final format: YYYY-MM-DDTHH:MM:SS."""
    if not dt_str or not isinstance(dt_str, str):
        return None
    if dt_str[-1] == 'Z':
        dt_str = dt_str[:-1]
    if len(dt_str) == 10:
        dt_str = dt_str + 'T00:00:00'
    elif len(dt_str) == 16 and dt_str[10] == 'T' and dt_str[13] == ':':
        dt_str = dt_str + ':00'
    return dt_str


def read_technical_pred(path):
    """DataFrame indexed like notebook 03's ``pd.read_json(..., orient='records')``.

    M³ base keeps coverage at the top level. M³ fast nests it under ``metadata``.
    A file whose status is anything other than success is a missing prediction.
    """
    with open(path, encoding='utf-8') as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict) or payload.get('status') not in (None, 'ok'):
        return pd.DataFrame(columns=['spatialCoverage', 'temporalCoverage'])
    if not isinstance(payload.get('spatialCoverage'), dict) and not isinstance(payload.get('temporalCoverage'), dict):
        metadata = payload.get('metadata')
        if isinstance(metadata, dict):
            payload = metadata
    frame = {}
    for column in ('spatialCoverage', 'temporalCoverage'):
        value = payload.get(column)
        if isinstance(value, dict):
            for key, item in value.items():
                frame.setdefault(key, {})[column] = item
    if not frame:
        return pd.DataFrame(columns=['spatialCoverage', 'temporalCoverage'])
    return pd.DataFrame.from_dict(frame, orient='index')


def _coverage_value(df_pred, key, column):
    if key not in df_pred.index or column not in df_pred.columns:
        return None
    value = df_pred.loc[key, column]
    if isinstance(value, (list, dict)):
        return None
    if pd.isna(value):
        return None
    return value


def locate_prediction(folder_pred, species_name):
    stem = species_name.replace(' ', '_')
    for name in (f'{stem}_inaturalist.json', f'{stem}_react.json', f'{stem}.json'):
        path = Path(folder_pred) / name
        if path.exists():
            return path
    return None


def load_pangaea_annotations():
    annotations = pd.read_csv(PANGAEA_ANNOT)
    cached = json.loads(PANGAEA_EXTENT_CACHE.read_text(encoding='utf-8'))
    if len(cached) != len(annotations):
        raise ValueError(f'Extent cache has {len(cached)} datasets, annotations have {len(annotations)}')
    annotations['spat_extent'] = annotations['id'].map(lambda i: cached[str(i)]['spat_extent'])
    annotations['temp_extent'] = annotations['id'].map(lambda i: cached[str(i)]['temp_extent'])
    return annotations


def naive_timestamp(value):
    """Parse one prediction timestamp as a naive clock time.

    A mixed column of date-only values, second precision, and fractional
    seconds makes ``pd.to_datetime`` drop the values that do not match the
    inferred format. Parsing each value keeps those timestamps.
    """
    text = fix_dt(value)
    if not isinstance(text, str):
        return pd.NaT
    if len(text) >= 6 and text[-6] in '+-' and text[-3] == ':':
        text = text[:-6]
    parsed = pd.to_datetime(text, errors='coerce')
    if pd.isna(parsed):
        return pd.NaT
    if getattr(parsed, 'tzinfo', None) is not None:
        parsed = parsed.replace(tzinfo=None)
    return parsed


def eval_spat_temp(folder_pred, folder_data):
    base_cols = ['min_lon', 'max_lon', 'min_lat', 'max_lat', 'min_time', 'max_time']
    df_metadata = {
        column: []
        for column in ['id'] + [f'{name}_annot' for name in base_cols] + [f'{name}_pred' for name in base_cols]
    }
    missing = []

    for filename in os.listdir(folder_data):
        if not filename.endswith('.csv'):
            continue
        species_name = filename.replace('.csv', '').replace('_', ' ')
        df_sel = pd.read_csv(os.path.join(folder_data, filename))
        lon = df_sel['decimalLongitude']
        lat = df_sel['decimalLatitude']
        df_sel['eventDate_fixed'] = df_sel['eventDate'].map(fix_dt)
        df_sel['datetime'] = pd.to_datetime(df_sel['eventDate_fixed'], errors='coerce')
        if df_sel['datetime'].isnull().any():
            bad = df_sel.loc[df_sel['datetime'].isnull(), 'eventDate'].tolist()
            raise ValueError(f'Unparseable eventDate in {filename}: {bad[:5]}')
        min_time, max_time = df_sel['datetime'].min(), df_sel['datetime'].max()

        df_metadata['id'].append(species_name)
        df_metadata['min_lon_annot'].append(lon.min())
        df_metadata['max_lon_annot'].append(lon.max())
        df_metadata['min_lat_annot'].append(lat.min())
        df_metadata['max_lat_annot'].append(lat.max())
        df_metadata['min_time_annot'].append(min_time)
        df_metadata['max_time_annot'].append(max_time)

        pred_path = locate_prediction(folder_pred, species_name)
        if pred_path is None:
            missing.append(species_name)
            for column in base_cols:
                df_metadata[f'{column}_pred'].append(None)
            continue

        df_pred = read_technical_pred(pred_path)
        df_metadata['min_lon_pred'].append(_coverage_value(df_pred, 'min_lon', 'spatialCoverage'))
        df_metadata['max_lon_pred'].append(_coverage_value(df_pred, 'max_lon', 'spatialCoverage'))
        df_metadata['min_lat_pred'].append(_coverage_value(df_pred, 'min_lat', 'spatialCoverage'))
        df_metadata['max_lat_pred'].append(_coverage_value(df_pred, 'max_lat', 'spatialCoverage'))
        df_metadata['min_time_pred'].append(fix_dt(_coverage_value(df_pred, 'from', 'temporalCoverage')))
        df_metadata['max_time_pred'].append(fix_dt(_coverage_value(df_pred, 'to', 'temporalCoverage')))

    df_metadata = pd.DataFrame(df_metadata)
    df_metadata['min_time_pred'] = df_metadata['min_time_pred'].map(naive_timestamp)
    df_metadata['max_time_pred'] = df_metadata['max_time_pred'].map(naive_timestamp)

    scores = {}
    for column in base_cols:
        scores[column] = float((df_metadata[f'{column}_pred'] == df_metadata[f'{column}_annot']).sum())
    scores['temp_average'] = float(np.mean([scores[column] for column in ['min_time', 'max_time']]))
    scores['spat_average'] = float(np.mean([scores[column] for column in ['min_lon', 'max_lon', 'min_lat', 'max_lat']]))
    scores['n_missing'] = len(missing)
    scores['n'] = len(df_metadata)
    return df_metadata, scores, missing


def eval_spat_temp_pangaea(folder_pred, df_metadata_annot, output_av_score_only=False):
    folder_pred = Path(folder_pred)
    if not folder_pred.is_dir():
        raise FileNotFoundError(folder_pred)
    base_cols = ['min_lon', 'max_lon', 'min_lat', 'max_lat', 'min_time', 'max_time']
    df_metadata = {
        column: []
        for column in ['id'] + [f'{name}_annot' for name in base_cols] + [f'{name}_pred' for name in base_cols]
    }
    ds_visited = set()
    for filename in os.listdir(folder_pred):
        if not filename.endswith('.json'):
            continue
        dataset_token = Path(filename).stem.split('_')[0]
        if not dataset_token.isdigit():
            continue
        ds_name = int(dataset_token)
        if ds_name in ds_visited:
            raise ValueError(f'Duplicate dataset {ds_name} in {folder_pred}')
        ds_visited.add(ds_name)
        ind_ds = df_metadata_annot[df_metadata_annot.id == ds_name].index
        if len(ind_ds) == 0:
            print(f'Dataset {ds_name} not found in annotations; skipping.')
            continue
        if len(ind_ds) != 1:
            raise ValueError(f'Multiple annotation rows for dataset {ds_name}')

        df_metadata['id'].append(ds_name)
        spat_extent = df_metadata_annot.loc[ind_ds, 'spat_extent'].values[0]
        if spat_extent is not None:
            min_lat, max_lat = spat_extent['southBoundLatitude'], spat_extent['northBoundLatitude']
            min_lon, max_lon = spat_extent['westBoundLongitude'], spat_extent['eastBoundLongitude']
        else:
            min_lon, max_lon, min_lat, max_lat = None, None, None, None
        temp_extent = df_metadata_annot.loc[ind_ds, 'temp_extent'].values[0]
        if temp_extent is not None:
            min_time, max_time = temp_extent[0], temp_extent[1]
        else:
            min_time, max_time = None, None

        df_metadata['min_lon_annot'].append(min_lon)
        df_metadata['max_lon_annot'].append(max_lon)
        df_metadata['min_lat_annot'].append(min_lat)
        df_metadata['max_lat_annot'].append(max_lat)
        df_metadata['min_time_annot'].append(min_time)
        df_metadata['max_time_annot'].append(max_time)

        df_pred = read_technical_pred(folder_pred / filename)
        if 'min_lon' in df_pred.index:
            df_metadata['min_lon_pred'].append(_coverage_value(df_pred, 'min_lon', 'spatialCoverage'))
            df_metadata['max_lon_pred'].append(_coverage_value(df_pred, 'max_lon', 'spatialCoverage'))
            df_metadata['min_lat_pred'].append(_coverage_value(df_pred, 'min_lat', 'spatialCoverage'))
            df_metadata['max_lat_pred'].append(_coverage_value(df_pred, 'max_lat', 'spatialCoverage'))
        else:
            df_metadata['min_lon_pred'].append(None)
            df_metadata['max_lon_pred'].append(None)
            df_metadata['min_lat_pred'].append(None)
            df_metadata['max_lat_pred'].append(None)
        df_metadata['min_time_pred'].append(fix_dt(_coverage_value(df_pred, 'from', 'temporalCoverage')))
        df_metadata['max_time_pred'].append(fix_dt(_coverage_value(df_pred, 'to', 'temporalCoverage')))

    ds_not_visited = set(df_metadata_annot.id.values) - ds_visited
    print(f'{folder_pred.name}: datasets not visited in predictions: {len(ds_not_visited)}')
    for ds_name in ds_not_visited:
        df_metadata['id'].append(ds_name)
        for column in base_cols:
            df_metadata[f'{column}_annot'].append(None)
            df_metadata[f'{column}_pred'].append(None)

    for key, values in df_metadata.items():
        if len(values) != len(df_metadata['id']):
            raise ValueError(f"Length mismatch for '{key}'")
    df_metadata = pd.DataFrame(df_metadata)
    for column in ('min_time_pred', 'max_time_pred'):
        df_metadata[column] = pd.to_datetime(
            df_metadata[column].apply(lambda value: value[:19] if isinstance(value, str) else None),
            errors='coerce',
        ).dt.tz_localize(None)
    for column in ('min_time_annot', 'max_time_annot'):
        df_metadata[column] = pd.to_datetime(df_metadata[column], errors='coerce').dt.tz_localize(None)
    for column in ('min_time_pred', 'max_time_pred', 'min_time_annot', 'max_time_annot'):
        years = df_metadata[column].dt.year
        df_metadata.loc[years.notna() & ~years.between(1, 9999), column] = pd.NaT
    for column in ('min_lat', 'max_lat', 'min_lon', 'max_lon'):
        df_metadata[f'{column}_pred'] = pd.to_numeric(df_metadata[f'{column}_pred'], errors='coerce')
        df_metadata[f'{column}_annot'] = pd.to_numeric(df_metadata[f'{column}_annot'], errors='coerce')
    df_metadata = df_metadata.sort_values('id').reset_index(drop=True)

    scores = {}
    for column in base_cols:
        inds_tp = pd.notna(df_metadata[f'{column}_annot']) & pd.notna(df_metadata[f'{column}_pred'])
        inds_fp = pd.isna(df_metadata[f'{column}_annot']) & pd.notna(df_metadata[f'{column}_pred'])
        inds_fn = pd.notna(df_metadata[f'{column}_annot']) & pd.isna(df_metadata[f'{column}_pred'])
        if 'time' in column:
            scores[column + '_accuracy'] = float(
                (df_metadata[f'{column}_pred'].dt.date[inds_tp] == df_metadata[f'{column}_annot'].dt.date[inds_tp]).sum()
            )
        else:
            scores[column + '_accuracy'] = float(
                np.sum(np.isclose(df_metadata[f'{column}_pred'][inds_tp], df_metadata[f'{column}_annot'][inds_tp], equal_nan=True))
            )
        both_or_pred = float(np.sum(inds_tp) + np.sum(inds_fp))
        both_or_annot = float(np.sum(inds_tp) + np.sum(inds_fn))
        scores[column + '_precision'] = float(np.sum(inds_tp)) / both_or_pred if both_or_pred else None
        scores[column + '_recall'] = float(np.sum(inds_tp)) / both_or_annot if both_or_annot else None

    def _average(values):
        numbers = [value for value in values if value is not None]
        return float(np.mean(numbers)) if numbers else None

    for metric in ('accuracy', 'precision', 'recall'):
        scores[f'temp_{metric}_average'] = _average([scores[f'min_time_{metric}'], scores[f'max_time_{metric}']])
        scores[f'spat_{metric}_average'] = _average([
            scores[f'min_lon_{metric}'], scores[f'max_lon_{metric}'],
            scores[f'min_lat_{metric}'], scores[f'max_lat_{metric}'],
        ])
    scores['n_missing'] = len(ds_not_visited)
    scores['n'] = len(df_metadata_annot)
    if output_av_score_only:
        kept = {key: value for key, value in scores.items() if key.endswith('_average') or key in ('n_missing', 'n')}
        return df_metadata, kept
    return df_metadata, scores


df_metadata_annot = load_pangaea_annotations()

pangaea_rows = []
inat_rows = []
technical_detail = {}
for spec in METHODS:
    _, pangaea_scores = eval_spat_temp_pangaea(spec['pangaea'], df_metadata_annot, output_av_score_only=True)
    _, inat_scores, _ = eval_spat_temp(spec['inaturalist'], INAT_DATA)
    technical_detail[(spec['method'], spec['llm'])] = {'pangaea': pangaea_scores, 'inaturalist': inat_scores}

    def _percent(value):
        return None if value is None else round(100 * value, 1)

    pangaea_rows.append({
        'Method': spec['method'],
        'LLM': spec['llm'],
        'n': pangaea_scores['n'],
        'Missing': pangaea_scores['n_missing'],
        'Spatial TP accuracy': round(pangaea_scores['spat_accuracy_average'], 1),
        'Spatial precision [%]': _percent(pangaea_scores['spat_precision_average']),
        'Spatial recall [%]': _percent(pangaea_scores['spat_recall_average']),
        'Temporal TP accuracy': round(pangaea_scores['temp_accuracy_average'], 1),
        'Temporal precision [%]': _percent(pangaea_scores['temp_precision_average']),
        'Temporal recall [%]': _percent(pangaea_scores['temp_recall_average']),
    })
    inat_rows.append({
        'Method': spec['method'],
        'LLM': spec['llm'],
        'n': inat_scores['n'],
        'Missing': inat_scores['n_missing'],
        'Spatial accuracy [%]': round(inat_scores['spat_average'], 1),
        'Temporal accuracy [%]': round(inat_scores['temp_average'], 1),
    })

technical_pangaea = pd.DataFrame(pangaea_rows)
technical_inat = pd.DataFrame(inat_rows)
technical_pangaea.to_csv(OUT / 'technical_pangaea.csv', index=False)
technical_inat.to_csv(OUT / 'technical_inaturalist.csv', index=False)

print('PANGAEA technical metadata')
display(technical_pangaea)
print('iNaturalist technical metadata')
display(technical_inat)


pangaea_inat_default-text-large: datasets not visited in predictions: 8
pangaea_inat_Qwen_Qwen3.6-35B-A3B-FP8: datasets not visited in predictions: 7
pangaea: datasets not visited in predictions: 0
pangaea: datasets not visited in predictions: 0
pangaea: datasets not visited in predictions: 0
pangaea: datasets not visited in predictions: 0
react_pangaea_default-text-large: datasets not visited in predictions: 0
react_pangaea_Qwen_Qwen3.6-35B-A3B-FP8: datasets not visited in predictions: 0
PANGAEA technical metadata


,Method,LLM,n,Missing,Spatial TP accuracy,Spatial precision [%],Spatial recall [%],Temporal TP accuracy,Temporal precision [%],Temporal recall [%]
0,M³ base,Llama3 70B,75,8,60.5,100.0,100.0,53.0,91.9,96.6
1,M³ base,Qwen 3.6 35B,75,7,60.5,97.1,100.0,53.0,86.4,96.6
2,M³ fast,Llama3 70B,75,0,60.5,100.0,97.1,53.0,95.0,93.4
3,M³ fast,Qwen 3.6 35B,75,0,59.5,97.1,97.1,49.0,96.5,90.2
4,M³ fast,Qwen 3.5 122B,75,0,59.5,100.0,95.6,49.5,96.6,91.8
5,M³ fast,Mistral Devstral 2 24B,75,0,60.0,97.0,95.6,53.0,92.1,95.1
6,ReAct,Llama3 70B,75,0,60.0,100.0,97.1,51.5,95.0,93.4
7,ReAct,Qwen 3.6 35B,75,0,59.5,98.5,95.6,49.5,91.9,93.4


iNaturalist technical metadata


,Method,LLM,n,Missing,Spatial accuracy [%],Temporal accuracy [%]
0,M³ base,Llama3 70B,100,0,99.8,82.5
1,M³ base,Qwen 3.6 35B,100,0,96.0,91.5
2,M³ fast,Llama3 70B,100,0,100.0,99.5
3,M³ fast,Qwen 3.6 35B,100,0,100.0,100.0
4,M³ fast,Qwen 3.5 122B,100,0,96.0,96.0
5,M³ fast,Mistral Devstral 2 24B,100,36,64.0,64.0
6,ReAct,Llama3 70B,100,0,100.0,94.0
7,ReAct,Qwen 3.6 35B,100,1,99.0,99.0


## Descriptive metadata

Faithfulness is the notebook 03 rubric (1–5, rescaled to 0–1). Similarity is factual-correctness precision of the predicted description against the PANGAEA reference description. The reported value is the mean ± standard error, in percent, over all 75 datasets. A dataset with no successful description scores zero.

M³ fast descriptions are judged here. Llama3 70B checkpoints in `outputs/m3_fast_vs_base/descriptive_judge` are reused when the metadata hash still matches. Qwen 3.6 35B checkpoints go in `descriptive_judge_qwen`, Qwen 3.5 122B in `descriptive_judge_qwen35`, and Mistral Devstral 2 24B in `descriptive_judge_devstral`. M³ base and ReAct reuse the columns already stored in `df_scores_pangaea_descriptive_with_baselines.csv`.


In [2]:
import hashlib

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import FactualCorrectness, RubricsScore
from tqdm.auto import tqdm

ENV_PATH = REPO / '.env'
load_dotenv(ENV_PATH, override=True)
SURF_API_KEY = os.getenv('SURF_API_KEY')
SURF_API_BASE = os.getenv('SURF_API_BASE') or 'https://willma.surf.nl/api/v0/'
if not SURF_API_KEY:
    raise RuntimeError(f'SURF_API_KEY is missing from {ENV_PATH}')

JUDGE_MODEL = 'mistralai/Devstral-Small-2-24B-Instruct-2512'
COL_NAMES = ['name', 'description', 'keywords']
FAST_RUNS = [
    {
        'llm': LLAMA,
        'pangaea': M3_ROOT / 'm3_fast_all_default-text-large_recovered' / 'm3' / 'pangaea',
        'judge_dir': OUT / 'descriptive_judge',
    },
    {
        'llm': QWEN,
        'pangaea': M3_ROOT / 'm3_fast_all_Qwen3.6-35B-A3B-FP8' / 'm3' / 'pangaea',
        'judge_dir': OUT / 'descriptive_judge_qwen',
    },
    {
        'llm': QWEN35,
        'pangaea': M3_ROOT / 'm3_fast_all_Qwen3.5-122B-A10B-NVFP4' / 'm3' / 'pangaea',
        'judge_dir': OUT / 'descriptive_judge_qwen35',
    },
    {
        'llm': DEVSTRAL,
        'pangaea': M3_ROOT / 'm3_fast_all_Devstral-Small-2-24B-Instruct-2512' / 'm3' / 'pangaea',
        'judge_dir': OUT / 'descriptive_judge_devstral',
    },
]
for spec in FAST_RUNS:
    spec['judge_dir'].mkdir(parents=True, exist_ok=True)
PANGAEA_DATA = REPO / 'outputs' / 'pangaea' / 'pangaea_datasets'

judge_llm = LangchainLLMWrapper(ChatOpenAI(
    model=JUDGE_MODEL,
    api_key=SURF_API_KEY,
    base_url=SURF_API_BASE,
    default_headers={'X-API-KEY': SURF_API_KEY},
    temperature=0,
    max_tokens=2048,
    timeout=600,
    max_retries=4,
))
grounding_scorer = RubricsScore(
    rubrics={
        'score1_description': 'Makes claims that contradict the data or reference fields/values that do not exist in it.',
        'score2_description': 'Several claims are unsupported by the data or appear invented.',
        'score3_description': 'Mostly supportable, but contains at least one unsupported or speculative claim.',
        'score4_description': 'Every claim is supportable from the data; some claims are vague or generic.',
        'score5_description': 'Every claim is directly supported by the data and is specific to it.',
    },
    llm=judge_llm,
)
similarity_scorer = FactualCorrectness(llm=judge_llm, mode='precision', atomicity='low')


def serialize_dataset(frame):
    schema = '\n'.join(f'- {column} ({dtype})' for column, dtype in frame.dtypes.items())
    sample = frame.head(5).to_markdown(index=False)
    return f'Columns:\n{schema}\n\nSample rows:\n{sample}\n\nRow count: {len(frame)}'


def metadata_sha256(metadata):
    encoded = json.dumps(metadata, sort_keys=True).encode()
    return hashlib.sha256(encoded).hexdigest()


def load_fast_record(folder, dataset_id):
    path = Path(folder) / f'{int(dataset_id)}.json'
    if not path.is_file():
        return None
    record = json.loads(path.read_text(encoding='utf-8'))
    if record.get('status') != 'ok' or not isinstance(record.get('metadata'), dict):
        return None
    return record


async def score_description(dataset_id, metadata, reference_description):
    data = pd.read_csv(PANGAEA_DATA / f'{int(dataset_id)}.csv')
    context = serialize_dataset(data)
    predicted = {key: metadata.get(key) for key in COL_NAMES}
    predicted['keywords'] = str(predicted['keywords'])
    if any(not isinstance(predicted[key], str) or not predicted[key].strip() for key in COL_NAMES):
        raise ValueError(f'Missing descriptive field for dataset {dataset_id}')
    abstract = '\n\n'.join(f'{key}: {predicted[key]}' for key in COL_NAMES)
    grounding_sample = SingleTurnSample(
        user_input=(
            'A model generated metadata for a dataset from the data alone.\n\n'
            f'DATASET:\n{context}\n\n'
            'Judge only whether the TITLE, DESCRIPTION, and KEYWORDS below are grounded in this data — '
            'whether its claims could be inferred from the data. Do not reward or '
            'penalize style, completeness, or omissions.'
        ),
        response=abstract,
    )
    similarity_sample = SingleTurnSample(
        response=predicted['description'],
        reference=reference_description,
    )
    rating = await grounding_scorer.single_turn_ascore(grounding_sample)
    similarity = await similarity_scorer.single_turn_ascore(similarity_sample)
    return (float(rating) - 1) / 4, float(similarity)


async def score_fast_descriptions(folder, judge_dir, llm_name):
    annotation_by_id = df_metadata_annot.set_index('id')
    pending = []
    for dataset_id in df_metadata_annot['id'].astype(int):
        record = load_fast_record(folder, dataset_id)
        if record is None:
            continue
        checkpoint = judge_dir / f'{dataset_id}.json'
        digest = metadata_sha256(record['metadata'])
        if checkpoint.is_file():
            saved = json.loads(checkpoint.read_text(encoding='utf-8'))
            if saved.get('status') == 'ok' and saved.get('metadata_sha256') == digest:
                continue
        pending.append((dataset_id, record, digest))

    print(f'M³ fast {llm_name}: {len(df_metadata_annot) - len(pending)} already scored, {len(pending)} still to judge')
    for dataset_id, record, digest in tqdm(pending, desc=f'M³ fast {llm_name}'):
        try:
            faithfulness, similarity = await score_description(
                dataset_id,
                record['metadata'],
                annotation_by_id.loc[dataset_id, 'description'],
            )
            if not (0 <= faithfulness <= 1 and 0 <= similarity <= 1):
                raise ValueError(f'Judge score outside 0-1: {faithfulness}, {similarity}')
            payload = {
                'dataset_id': int(dataset_id),
                'status': 'ok',
                'metadata_sha256': digest,
                'judge_model': JUDGE_MODEL,
                'faithfulness': faithfulness,
                'similarity': similarity,
            }
        except Exception as exc:
            payload = {
                'dataset_id': int(dataset_id),
                'status': 'judge_error',
                'error': f'{type(exc).__name__}: {exc}',
            }
            print(f'Row {dataset_id} failed: {payload["error"]}')
        checkpoint = judge_dir / f'{dataset_id}.json'
        temporary = checkpoint.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(payload), encoding='utf-8')
        temporary.replace(checkpoint)


def score_label(series):
    mean = round(100 * series.mean(), 1)
    sem = round(100 * series.sem(), 1)
    return f'{mean} ± {sem}'


def fast_descriptive_frame(folder, judge_dir, llm_name):
    rows = []
    failures = []
    for dataset_id in df_metadata_annot['id'].astype(int):
        record = load_fast_record(folder, dataset_id)
        if record is None:
            rows.append({'id': dataset_id, 'faithfulness': 0.0, 'similarity': 0.0})
            continue
        checkpoint = judge_dir / f'{dataset_id}.json'
        saved = json.loads(checkpoint.read_text(encoding='utf-8')) if checkpoint.is_file() else {}
        if saved.get('status') != 'ok' or saved.get('metadata_sha256') != metadata_sha256(record['metadata']):
            failures.append(dataset_id)
            continue
        rows.append({
            'id': dataset_id,
            'faithfulness': float(saved['faithfulness']),
            'similarity': float(saved['similarity']),
        })
    if failures:
        raise RuntimeError(
            f'M³ fast {llm_name} still has {len(failures)} unscored descriptions, including {failures[:5]}. '
            'The descriptive table was not updated.'
        )
    return pd.DataFrame(rows)


for spec in FAST_RUNS:
    await score_fast_descriptions(spec['pangaea'], spec['judge_dir'], spec['llm'])

saved_scores = pd.read_csv(DESCRIPTIVE_SCORES).set_index('id')
dataset_ids = df_metadata_annot['id'].astype(int)
saved_runs = [
    ('M³ base', LLAMA, 'faith_llama3_70B', 'sim_llama3_70B'),
    ('M³ base', QWEN, 'faith_Qwen_3.6_35B', 'sim_Qwen_3.6_35B'),
    ('ReAct', LLAMA, 'faith_ReAct_default-text-large', 'sim_ReAct_default-text-large'),
    ('ReAct', QWEN, 'faith_ReAct_Qwen_Qwen3.6-35B-A3B-FP8', 'sim_ReAct_Qwen_Qwen3.6-35B-A3B-FP8'),
]
missing_columns = [column for _, _, faith, sim in saved_runs for column in (faith, sim) if column not in saved_scores.columns]
if missing_columns:
    raise ValueError(f'Missing descriptive columns: {missing_columns}')
if saved_scores.loc[dataset_ids, [column for _, _, faith, sim in saved_runs for column in (faith, sim)]].isna().any().any():
    raise ValueError('Saved descriptive scores have missing values')

descriptive_per_dataset = pd.DataFrame({'id': dataset_ids.to_numpy()})
for method, llm, faith_column, sim_column in saved_runs:
    key = f'{method}|{llm}'
    descriptive_per_dataset[f'faith_{key}'] = saved_scores.loc[dataset_ids, faith_column].to_numpy()
    descriptive_per_dataset[f'sim_{key}'] = saved_scores.loc[dataset_ids, sim_column].to_numpy()
for spec in FAST_RUNS:
    fast_scores = fast_descriptive_frame(spec['pangaea'], spec['judge_dir'], spec['llm'])
    key = f"M³ fast|{spec['llm']}"
    descriptive_per_dataset = descriptive_per_dataset.merge(
        fast_scores.rename(columns={'faithfulness': f'faith_{key}', 'similarity': f'sim_{key}'}),
        on='id',
        how='left',
    )
descriptive_per_dataset.to_csv(OUT / 'descriptive_pangaea_per_dataset.csv', index=False)

descriptive_rows = []
for spec in METHODS:
    key = f"{spec['method']}|{spec['llm']}"
    descriptive_rows.append({
        'Method': spec['method'],
        'LLM': spec['llm'],
        'Faithfulness [%]': score_label(descriptive_per_dataset[f'faith_{key}']),
        'Similarity [%]': score_label(descriptive_per_dataset[f'sim_{key}']),
    })
descriptive_pangaea = pd.DataFrame(descriptive_rows)
descriptive_pangaea.to_csv(OUT / 'descriptive_pangaea.csv', index=False)
print('PANGAEA descriptive metadata')
display(descriptive_pangaea)


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


M³ fast Llama3 70B: 75 already scored, 0 still to judge


/tmp/ipykernel_83887/166641445.py:7: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, RubricsScore
/tmp/ipykernel_83887/166641445.py:7: DeprecationWarning: Importing RubricsScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import RubricsScore
  from ragas.metrics import FactualCorrectness, RubricsScore
/tmp/ipykernel_83887/166641445.py:45: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOpenAI(


M³ fast Llama3 70B: 0it [00:00, ?it/s]

M³ fast Qwen 3.6 35B: 75 already scored, 0 still to judge


M³ fast Qwen 3.6 35B: 0it [00:00, ?it/s]

... (2 outputs left out)

M³ fast Mistral Devstral 2 24B: 0 already scored, 75 still to judge


M³ fast Mistral Devstral 2 24B:   0%|          | 0/75 [00:00<?, ?it/s]

PANGAEA descriptive metadata


,Method,LLM,Faithfulness [%],Similarity [%]
0,M³ base,Llama3 70B,94.7 ± 1.7,56.0 ± 3.4
1,M³ base,Qwen 3.6 35B,95.3 ± 2.1,45.1 ± 3.6
2,M³ fast,Llama3 70B,97.0 ± 1.3,20.6 ± 2.6
3,M³ fast,Qwen 3.6 35B,96.3 ± 1.6,28.1 ± 3.0
4,M³ fast,Qwen 3.5 122B,95.7 ± 1.7,33.3 ± 3.1
5,M³ fast,Mistral Devstral 2 24B,96.0 ± 1.4,46.1 ± 3.2
6,ReAct,Llama3 70B,98.7 ± 1.0,28.5 ± 3.0
7,ReAct,Qwen 3.6 35B,89.7 ± 2.6,38.2 ± 3.2


## All scores

PANGAEA joins the technical scores with faithfulness and similarity. iNaturalist lists the technical scores, which are the full score set for that corpus. Both tables include the model in the LLM column.


In [3]:
all_pangaea = technical_pangaea.merge(descriptive_pangaea, on=['Method', 'LLM'], how='left')
all_pangaea.to_csv(OUT / 'all_scores_pangaea.csv', index=False)
technical_inat.to_csv(OUT / 'all_scores_inaturalist.csv', index=False)

print('PANGAEA')
display(all_pangaea)
print('iNaturalist')
display(technical_inat)


PANGAEA


,Method,LLM,n,Missing,Spatial TP accuracy,Spatial precision [%],Spatial recall [%],Temporal TP accuracy,Temporal precision [%],Temporal recall [%],Faithfulness [%],Similarity [%]
0,M³ base,Llama3 70B,75,8,60.5,100.0,100.0,53.0,91.9,96.6,94.7 ± 1.7,56.0 ± 3.4
1,M³ base,Qwen 3.6 35B,75,7,60.5,97.1,100.0,53.0,86.4,96.6,95.3 ± 2.1,45.1 ± 3.6
2,M³ fast,Llama3 70B,75,0,60.5,100.0,97.1,53.0,95.0,93.4,97.0 ± 1.3,20.6 ± 2.6
3,M³ fast,Qwen 3.6 35B,75,0,59.5,97.1,97.1,49.0,96.5,90.2,96.3 ± 1.6,28.1 ± 3.0
4,M³ fast,Qwen 3.5 122B,75,0,59.5,100.0,95.6,49.5,96.6,91.8,95.7 ± 1.7,33.3 ± 3.1
5,M³ fast,Mistral Devstral 2 24B,75,0,60.0,97.0,95.6,53.0,92.1,95.1,96.0 ± 1.4,46.1 ± 3.2
6,ReAct,Llama3 70B,75,0,60.0,100.0,97.1,51.5,95.0,93.4,98.7 ± 1.0,28.5 ± 3.0
7,ReAct,Qwen 3.6 35B,75,0,59.5,98.5,95.6,49.5,91.9,93.4,89.7 ± 2.6,38.2 ± 3.2


iNaturalist


,Method,LLM,n,Missing,Spatial accuracy [%],Temporal accuracy [%]
0,M³ base,Llama3 70B,100,0,99.8,82.5
1,M³ base,Qwen 3.6 35B,100,0,96.0,91.5
2,M³ fast,Llama3 70B,100,0,100.0,99.5
3,M³ fast,Qwen 3.6 35B,100,0,100.0,100.0
4,M³ fast,Qwen 3.5 122B,100,0,96.0,96.0
5,M³ fast,Mistral Devstral 2 24B,100,36,64.0,64.0
6,ReAct,Llama3 70B,100,0,100.0,94.0
7,ReAct,Qwen 3.6 35B,100,1,99.0,99.0
